# 元编程与描述符协议 (Metaprogramming & Descriptors)

> 本 Notebook 展示了 Python 底层对象创建、属性拦截和元编程的高级用法。

## 1. 线程安全单例元类
利用元类的 `__call__` 拦截实例创建过程。

In [ ]:
import threading
from typing import Any, Dict, Type

class SingletonMeta(type):
    _instances: Dict[Type[Any], Any] = {}
    _lock: threading.Lock = threading.Lock()

    def __call__(cls, *args: Any, **kwargs: Any) -> Any:
        if cls not in cls._instances:
            with cls._lock:
                if cls not in cls._instances:
                    instance = super().__call__(*args, **kwargs)
                    cls._instances[cls] = instance
        return cls._instances[cls]

class DatabaseConnection(metaclass=SingletonMeta):
    def __init__(self) -> None:
        self.connected = True

db1 = DatabaseConnection()
db2 = DatabaseConnection()
print(f"db1 is db2: {db1 is db2}")

## 2. 描述符协议实现属性校验
通过 `__get__` 和 `__set__` 代理属性访问。

In [ ]:
class IntegerField:
    def __init__(self, min_val: int = 0) -> None:
        self.min_val = min_val

    def __set_name__(self, owner: Any, name: str) -> None:
        self.storage_name = f"_{name}"

    def __get__(self, instance: Any, owner: Any) -> Any:
        if instance is None: return self
        return getattr(instance, self.storage_name, None)

    def __set__(self, instance: Any, value: int) -> None:
        if not isinstance(value, int):
            raise TypeError(f"Expected int, got {type(value).__name__}")
        if value < self.min_val:
            raise ValueError(f"Value must be >= {self.min_val}")
        setattr(instance, self.storage_name, value)

class Product:
    price = IntegerField(min_val=1)

p = Product(100)
print(f"Price: {p.price}")
try:
    p.price = -1
except ValueError as e:
    print(f"Caught expected error: {e}")

## 3. 性能测试
对比普通访问与描述符访问。

In [ ]:
import timeit

setup = """
class Normal:
    def __init__(self): self.val = 1
class Descriptor:
    def __init__(self, v): self.v = v
    def __get__(self, i, o): return self.v
class DescObj:
    val = Descriptor(1)
n = Normal()
d = DescObj()
"""

normal_time = timeit.timeit("n.val", setup=setup, number=1000000)
desc_time = timeit.timeit("d.val", setup=setup, number=1000000)
print(f"Normal access: {normal_time:.4f}s")
print(f"Descriptor access: {desc_time:.4f}s")